In [4]:
#Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.optimize import least_squares
import pandas as pd

# ============================================================
# Scaling parameters
# ============================================================
A_S = 0.065
A_T = 0.72
ALPHA = 0.305

# ============================================================
# Scaling function
# ============================================================
def compute_tstarstar(t_GeV2, sqrts_TeV, a_s=A_S, a_t=A_T):
    """
    Compute t** from |t| using the scaling formula:
    t** = (s/1TeV²)^0.065 * (|t|/1GeV²)^0.72
    
    Args:
        t_GeV2: |t| in GeV²
        sqrts_TeV: sqrt(s) in TeV
        a_s: exponent for s (default 0.065)
        a_t: exponent for t (default 0.72)
    
    Returns:
        t**: scaled variable
    """
    s_TeV2 = sqrts_TeV**2
    tstarstar = np.power(s_TeV2, a_s) * np.power(t_GeV2, a_t)
    return tstarstar

def compute_scaled_y(dsigma_dt, sqrts_TeV, alpha=ALPHA):
    """
    Compute scaled cross section:
    y = (s/1TeV²)^(-α) * dσ/dt
    
    Args:
        dsigma_dt: dσ/dt in mb/GeV²
        sqrts_TeV: sqrt(s) in TeV
        alpha: scaling exponent (default 0.305)
    
    Returns:
        y: scaled cross section
    """
    s_TeV2 = sqrts_TeV**2
    y = dsigma_dt / np.power(s_TeV2, alpha)
    return y

# ============================================================
# Main scaling function
# ============================================================
def scale_data(input_csv, output_csv, sqrts_TeV):
    """
    Read differential cross section data and apply scaling.
    
    Args:
        input_csv: path to input CSV with columns: t_c, t_star, dsigma_dt, dsigma_dt_err
        output_csv: path to output CSV
        sqrts_TeV: center-of-mass energy in TeV
    """
    # Read data
    df = pd.read_csv(input_csv)
    
    print(f"\nScaling data from {input_csv}")
    print(f"√s = {sqrts_TeV} TeV")
    print(f"Number of data points: {len(df)}")
    print(f"\nScaling formula:")
    print(f"  t** = (s/1TeV²)^{A_S} * (|t|/1GeV²)^{A_T}")
    print(f"  y = (s/1TeV²)^(-{ALPHA}) * dσ/dt")
    
    # Use t_star (corrected position) for scaling
    t_values = df['t_star'].values
    dsigma_dt = df['dsigma_dt'].values
    dsigma_dt_err = df['dsigma_dt_err'].values
    
    # Compute scaled variables
    tstarstar = compute_tstarstar(t_values, sqrts_TeV)
    y = compute_scaled_y(dsigma_dt, sqrts_TeV)
    y_err = compute_scaled_y(dsigma_dt_err, sqrts_TeV)
    
    # Create output dataframe
    df_scaled = pd.DataFrame({
        'sqrts_TeV': sqrts_TeV,
        't_star': t_values,
        'tstarstar': tstarstar,
        'y': y,
        'y_err': y_err,
        'dsigma_dt': dsigma_dt,
        'dsigma_dt_err': dsigma_dt_err
    })
    
    # Save to CSV
    df_scaled.to_csv(output_csv, index=False, float_format='%.8f')
    
    print(f"\nScaled data saved to: {output_csv}")
    print(f"\nOutput columns:")
    print(f"  sqrts_TeV: center-of-mass energy (TeV)")
    print(f"  t_star: |t*| (GeV²)")
    print(f"  tstarstar: t** scaled variable")
    print(f"  y: scaled dσ/dt")
    print(f"  y_err: uncertainty on y")
    print(f"  dsigma_dt: original dσ/dt (mb/GeV²)")
    print(f"  dsigma_dt_err: original uncertainty (mb/GeV²)")
    
    print(f"\nFirst few rows:")
    print(df_scaled.head())
    
    print(f"\nt** range: [{tstarstar.min():.4f}, {tstarstar.max():.4f}]")
    print(f"y range: [{y.min():.6f}, {y.max():.6f}]")
    
    return df_scaled


# ============================================================
# Example usage
# ============================================================
if __name__ == "__main__":
    # D0 data at sqrt(s) = 1.96 TeV
    input_file = "/home/jesusavc/phd/odderon/scaling/D0_data_ppbar.csv"
    output_file = "/home/jesusavc/phd/odderon/scaling/scaled_data/D0_data_1.96TeV_scaled.csv"
    sqrts = 1.96  # TeV
    
    df_scaled = scale_data(input_file, output_file, sqrts)


Scaling data from /home/jesusavc/phd/odderon/scaling/D0_data_ppbar.csv
√s = 1.96 TeV
Number of data points: 17

Scaling formula:
  t** = (s/1TeV²)^0.065 * (|t|/1GeV²)^0.72
  y = (s/1TeV²)^(-0.305) * dσ/dt

Scaled data saved to: /home/jesusavc/phd/odderon/scaling/scaled_data/D0_data_1.96TeV_scaled.csv

Output columns:
  sqrts_TeV: center-of-mass energy (TeV)
  t_star: |t*| (GeV²)
  tstarstar: t** scaled variable
  y: scaled dσ/dt
  y_err: uncertainty on y
  dsigma_dt: original dσ/dt (mb/GeV²)
  dsigma_dt_err: original uncertainty (mb/GeV²)

First few rows:
   sqrts_TeV  t_star  tstarstar         y     y_err  dsigma_dt  dsigma_dt_err
0       1.96   0.258   0.411490  3.137509  0.172463      4.730          0.260
1       1.96   0.298   0.456488  1.419507  0.047759      2.140          0.072
2       1.96   0.338   0.499820  0.703120  0.024543      1.060          0.037
3       1.96   0.378   0.541736  0.374113  0.014593      0.564          0.022
4       1.96   0.418   0.582425  0.197670  0.00